Projection Montage.  
Requires pre-computed `.data` files in `image_results/` (torch-saved numpy arrays).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import torch

TEXTWIDTH = 4.80

plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['STIXGeneral', 'DejaVu Serif', 'Liberation Serif'],
    'mathtext.fontset':  'stix',
    'font.size':          8,
    'axes.titlesize':     9,
    'axes.titleweight':  'bold',
    'axes.labelsize':     8.5,
    'xtick.labelsize':    7,
    'ytick.labelsize':    7,
    'legend.fontsize':    6.5,
    'figure.dpi':         150,
    'savefig.dpi':        300,
    'savefig.bbox':      'tight',
    'savefig.pad_inches': 0.03,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
})

SAVE = '../images'
os.makedirs(SAVE, exist_ok=True)

In [ ]:
def load_volumes(names):
    volumes = {}
    for name in names:
        if os.path.exists(name):
            key = name.split('/')[-1].replace('.data', '')
            volumes[key] = torch.load(name, weights_only=False)
    return volumes


def compute_projections(vol, coords):
    z, x, y = coords
    axial    = vol[z, :, :]
    sagittal = vol[:, x, :]
    coronal  = vol[:, :, y]
    return axial, sagittal[::-1], coronal[::-1]

Each `.data` file contains a 3D numpy array (Z, H, W) in HU scale [-1000, 1000],  
saved with `torch.save()` during inference.

In [ ]:
DATA_DIR = './image_results'

vols = load_volumes([
    f'{DATA_DIR}/Smile.data',
    f'{DATA_DIR}/reg_SegResNet.data',
    f'{DATA_DIR}/DDPM.data',
    f'{DATA_DIR}/flow_TimeResNet.data',
    f'{DATA_DIR}/Native.data',
    f'{DATA_DIR}/Contrast.data',
])

if 'DDPM' in vols and hasattr(vols['DDPM'], 'numpy'):
    vols['DDPM'] = vols['DDPM'].numpy()

for key in vols:
    print(key, vols[key].shape)

In [ ]:
def create_projection_montage(volumes, output_path, coords=None, dpi=150):
    names = list(volumes.keys())
    n = len(names)
    fig, axes = plt.subplots(nrows=n, ncols=6,
                             figsize=(24, 3 * n),
                             constrained_layout=False)
    fig.subplots_adjust(left=0.07, right=0.98, top=0.98, bottom=0.01,
                        hspace=0.05, wspace=0.05)

    col_titles = ['Axial', 'Sagittal', 'Coronal',
                  'Diff (Axial)', 'Diff (Sagittal)', 'Diff (Coronal)']
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=14)

    nat_axial, nat_sagittal, nat_coronal = compute_projections(
        volumes['Native'], coords)

    for i, name in enumerate(names):
        vol = volumes[name]
        axial, sagittal, coronal = compute_projections(vol, coords)

        for j, img in enumerate((axial, sagittal, coronal)):
            ax = axes[i, j]
            ax.imshow(img.clip(-400, 200), cmap='gray')
            ax.axis('off')

        maxabs = 300
        for j, (img, img_nat) in enumerate(
                zip((axial, sagittal, coronal),
                    (nat_axial, nat_sagittal, nat_coronal)), start=3):
            ax = axes[i, j]
            if name in ['Native', 'Contrast']:
                img = img.clip(-1000, 1000)
            ax.imshow(img - img_nat.clip(-1000, 1000),
                      cmap='seismic', vmin=-maxabs, vmax=maxabs)
            ax.axis('off')

        y_pos = 1 - (i + 0.5) / n
        fig.text(0.05, y_pos, name, rotation='vertical',
                 va='center', ha='center', fontsize=12)

    fig.savefig(output_path, dpi=dpi)
    print(f'Saved: {output_path}')
    plt.show()


z = vols['Native'].shape[0] // 2
y = vols['Native'].shape[1] // 2
x = vols['Native'].shape[2] // 2
create_projection_montage(vols, f'{SAVE}/projection_montage.png', coords=(z, y, x))